### Refactored Triggers for per user schema tracking

In [2]:
"""
// ON CREATE TRIGGER
{

  CREATE TRIGGER schema_oncreate_trigger ON CREATE
  AFTER COMMIT EXECUTE
  WITH createdVertices, createdEdges

  // create schema nodes for the created labels.
  CALL {
    WITH createdVertices
    MERGE (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH createdVertices, schema
    UNWIND createdVertices AS newNodes
    UNWIND labels(newNodes) AS addedLabel

    WITH addedLabel, count(*) AS num_occurances, schema
    MERGE (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: addedLabel})
    SET schema_node.count = coalesce(schema_node.count, 0) + num_occurances
  }

  // Create properties for each label based on newly created nodes.
  CALL {
    WITH createdVertices

    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    UNWIND createdVertices AS newNode
    WITH newNode, labels(newNode) AS nodeLabels, keys(newNode) AS nodePropertyNames, schema

    // group keys and values to set for a label across all created nodes.
    UNWIND nodeLabels AS nodeLabel
    UNWIND nodePropertyNames AS propertyName
    WITH nodeLabel, propertyName, valuetype(newNode[propertyName]) AS propertyType, schema

    // For each label and key, get the first type for that property key because across all nodes they might have same label, same key, but different property type (very unlikely tho).
    WITH nodeLabel, propertyName, count(*) AS propertyCount, head(collect(propertyType)) AS firstPropertyType, schema

    // Group properties, types, and their counts by label.
    WITH nodeLabel, collect(propertyName) AS propertyNames, collect(firstPropertyType) AS propertyTypes, collect(propertyCount) AS propertyCounts, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: nodeLabel})

    // Iterate over each property to create / increase count.
    UNWIND range(0, size(propertyNames) - 1) AS index
    MERGE (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: propertyNames[index]})
    
    // Update the count of occurrences for the property.
    // If the property type is not already set, assign the detected type.
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) + propertyCounts[index], 
        schema_node_prop.type = coalesce(schema_node_prop.type, propertyTypes[index])
  }

  // create schema relationships for the created edges.
  CALL {
    WITH createdEdges
    MERGE (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH createdEdges, schema
    UNWIND createdEdges AS newEdge

    WITH type(newEdge) AS edgeType, schema
    WITH edgeType, count(*) as num_occurances, schema

    MERGE (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel :SchemaRelationship{type: edgeType})
    SET schema_rel.count = coalesce(schema_rel.count, 0) + num_occurances
  }

  // Create properities for each type based on newly created edges.
  CALL {
    WITH createdEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    UNWIND createdEdges AS newEdge
    WITH newEdge, type(newEdge) AS relationshipType, keys(newEdge) AS relationshipPropertyNames, schema

    // group keys and values to set for a type across all created edges.
    UNWIND relationshipPropertyNames AS propertyName
    WITH relationshipType, propertyName, valuetype(newEdge[propertyName]) AS propertyType, schema

    // For each type and key, get the first type for that property key because across all edges they might have same type, same key, but different property type (very unlikely tho).
    WITH relationshipType, propertyName, count(*) AS propertyCount, head(collect(propertyType)) AS firstPropertyType, schema

    // Group properties, their data types, and their counts by relationship type.
    WITH relationshipType, collect(propertyName) AS propertyNames, collect(firstPropertyType) AS propertyTypes, collect(propertyCount) AS propertyCounts, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel :SchemaRelationship {type: relationshipType})

    UNWIND range(0, size(propertyNames) - 1) AS index
    MERGE (schema_rel)-[:IS_SCHEMA_RELATIONSHIP_PROPERTY]->(schema_rel_prop :SchemaRelationshipProperty{property_name: propertyNames[index]})
    
    // Update the count of occurrences for the property.
    // If the property type is not already set, assign the detected type.
    SET schema_rel_prop.count = coalesce(schema_rel_prop.count, 0) + propertyCounts[index], 
        schema_rel_prop.type = coalesce(schema_rel_prop.type, propertyTypes[index])
  }

  // Create schema out_relationships for each label based on newly created edges.
  CALL {
    WITH createdEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH createdEdges, schema
    UNWIND createdEdges AS newEdge

    WITH startNode(newEdge) AS startNode, type(newEdge) AS relationshipType, endNode(newEdge) AS endNode, schema
    WITH startNode, relationshipType, count(*) AS outgoingRelationshipCount, collect(endNode) AS endNodes, schema

    UNWIND endNodes AS endNode
    UNWIND labels(endNode) AS endNodeLabel
    WITH startNode, relationshipType, outgoingRelationshipCount, collect(endNodeLabel) AS endNodeLabels, schema

    UNWIND labels(startNode) AS label
    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: label})

    MERGE (schema_node)-[:IS_SCHEMA_NODE_OUT_REL]->(schema_node_out_rel :SchemaNodeOutRelationship{rel_type: relationshipType})

    // Update the relationship count and unique destination labels.
    SET schema_node_out_rel.count = coalesce(schema_node_out_rel.count, 0) + outgoingRelationshipCount, 
        schema_node_out_rel.to_labels = collections.union(coalesce(schema_node_out_rel.to_labels, []), endNodeLabels)
  }

}
"""

"\n// ON CREATE TRIGGER\n{\n\n  CREATE TRIGGER schema_oncreate_trigger ON CREATE\n  AFTER COMMIT EXECUTE\n  WITH createdVertices, createdEdges\n\n  // create schema nodes for the created labels.\n  CALL {\n    WITH createdVertices\n    MERGE (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})\n\n    WITH createdVertices, schema\n    UNWIND createdVertices AS newNodes\n    UNWIND labels(newNodes) AS addedLabel\n\n    WITH addedLabel, count(*) AS num_occurances, schema\n    MERGE (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: addedLabel})\n    SET schema_node.count = coalesce(schema_node.count, 0) + num_occurances\n  }\n\n  // Create properties for each label based on newly created nodes.\n  CALL {\n    WITH createdVertices\n\n    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})\n\n    UNWIND createdVertices AS newNode\n    WITH newNode, labels(newNode) AS nodeLabels, keys(newNode) AS nodePropertyNames, schema\n\n    // group keys and values to set for a label acros

In [3]:
"""
// ON UPDATE TRIGGER
{

  CREATE TRIGGER schema_onupdate_trigger ON UPDATE
  AFTER COMMIT EXECUTE
  WITH removedVertexProperties, removedEdgeProperties, setVertexLabels, removedVertexLabels

  // Reduce count of schema label properties
  CALL {
    WITH removedVertexProperties
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH removedVertexProperties, schema
    UNWIND removedVertexProperties AS removedVertexProperty

    WITH removedVertexProperty.vertex AS node, removedVertexProperty.key AS removedPropertyName, schema
    UNWIND labels(node) AS nodeLabel
    WITH nodeLabel, collect(removedPropertyName) as removedPropertyNames, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: nodeLabel})
    UNWIND removedPropertyNames AS removedPropertyName
    
    MATCH (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: removedPropertyName})
    // reduce the count of that schema_node_prop
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) - 1
  }

  // Reduce count of schema relationship properties
  CALL {
    WITH removedEdgeProperties
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH removedEdgeProperties, schema
    UNWIND removedEdgeProperties AS removedEdgeProperty

    WITH removedEdgeProperty.edge AS edge, removedEdgeProperty.key AS removedPropertyName, schema
    WITH type(edge) AS edgeType, collect(removedPropertyName) AS removedPropertyNames, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel :SchemaRelationship{type: edgeType})
    UNWIND removedPropertyNames AS removedPropertyName
    
    MATCH (schema_rel)-[:IS_SCHEMA_RELATIONSHIP_PROPERTY]->(schema_rel_prop :SchemaRelationshipProperty{property_name: removedPropertyName})
    SET schema_rel_prop.count = coalesce(schema_rel_prop.count, 0) - 1
  }

  // Delete schema label property if they are less than 1 from the reductions above (zero)
  CALL {
    WITH removedVertexProperties
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH removedVertexProperties, schema
    UNWIND removedVertexProperties AS removedVertexProperty
    WITH removedVertexProperty.vertex AS node, schema
    UNWIND labels(node) AS editedNodeLabel
    WITH DISTINCT editedNodeLabel, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(:SchemaNode {label: editedNodeLabel})-[:IS_SCHEMA_NODE_PROPERTY]->(property)
    WHERE property.count < 1
    DETACH DELETE property
  }

  // Delete schema relationship property if they are less than 1 from the reductions above (zero)
  CALL {
    WITH removedEdgeProperties
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH removedEdgeProperties, schema
    UNWIND removedEdgeProperties AS removedEdgeProperty
    WITH type(removedEdgeProperty.edge) AS edgeType, schema
    WITH DISTINCT edgeType, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(:SchemaRelationship{type: edgeType})-[:IS_SCHEMA_RELATIONSHIP_PROPERTY]->(property)
    WHERE property.count < 1
    DETACH DELETE property
  }

  // when additional label was set to vertex
  CALL {
    WITH setVertexLabels
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH setVertexLabels, schema
    UNWIND setVertexLabels AS labelSetOnVertex
    WITH labelSetOnVertex.label AS addedLabel, labelSetOnVertex.vertices AS listOfVertices, schema
    UNWIND listOfVertices AS singleVertex

    MERGE (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: addedLabel})
    SET schema_node.count = coalesce(schema_node.count, 0) + 1

    WITH addedLabel, singleVertex, schema_node
    UNWIND keys(singleVertex) as propertyName

    WITH addedLabel, singleVertex, propertyName, valueType(singleVertex[propertyName]) AS propertyType, schema_node
    MERGE (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: propertyName})
    // Only set property type if there is no type already (first time creating that property.)
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) + 1, 
        schema_node_prop.type = coalesce(schema_node_prop.type, propertyType)
  }

  // when a label was removed from vertex
  CALL {
    WITH removedVertexLabels
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    
    WITH removedVertexLabels, schema
    UNWIND removedVertexLabels AS removedVertexLabel
    WITH removedVertexLabel.label AS removedLabel, removedVertexLabel.vertices AS listOfVertices, schema
    UNWIND listOfVertices AS singleVertex

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: removedLabel})
    SET schema_node.count = schema_node.count - 1

    WITH removedLabel, singleVertex, schema_node
    UNWIND keys(singleVertex) as propertyName

    WITH removedLabel, singleVertex, propertyName, valueType(singleVertex[propertyName]) AS propertyType, schema_node
    MATCH (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: propertyName})
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) - 1
  }

  // Delete label, properities & out_relationships if the label count is less than 1 (zero)
  CALL {
    WITH removedVertexLabels
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    
    WITH removedVertexLabels, schema
    UNWIND removedVertexLabels AS removedVertexLabel
    WITH removedVertexLabel.label AS removedLabel, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: removedLabel})
    WHERE schema_node.count < 1
    OPTIONAL MATCH (schema_node)-->(prop_or_rel) 
    DETACH DELETE prop_or_rel, schema_node
  }    

  // Delete label property if removing the label from a node reduced the count below 1 (zero)
  CALL {
    WITH removedVertexLabels
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    
    WITH removedVertexLabels, schema
    UNWIND removedVertexLabels AS removedVertexLabel
    WITH removedVertexLabel.label AS removedLabel, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node {label: removedLabel})-[:IS_SCHEMA_NODE_PROPERTY]->(prop) 
    WHERE prop.count < 1
    DETACH DELETE prop
  }

}
"""

"\n// ON UPDATE TRIGGER\n{\n\n  CREATE TRIGGER schema_onupdate_trigger ON UPDATE\n  AFTER COMMIT EXECUTE\n  WITH removedVertexProperties, removedEdgeProperties, setVertexLabels, removedVertexLabels\n\n  // Reduce count of schema label properties\n  CALL {\n    WITH removedVertexProperties\n    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})\n\n    WITH removedVertexProperties, schema\n    UNWIND removedVertexProperties AS removedVertexProperty\n\n    WITH removedVertexProperty.vertex AS node, removedVertexProperty.key AS removedPropertyName, schema\n    UNWIND labels(node) AS nodeLabel\n    WITH nodeLabel, collect(removedPropertyName) as removedPropertyNames, schema\n\n    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: nodeLabel})\n    UNWIND removedPropertyNames AS removedPropertyName\n    \n    MATCH (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop :SchemaNodeProperty{property_name: removedPropertyName})\n    // reduce the count of that schema_

In [4]:
"""
// ON DELETE TRIGGER
{

  CREATE TRIGGER schema_ondelete_trigger ON DELETE
  AFTER COMMIT EXECUTE
  WITH deletedVertices, deletedEdges

  // reduce label properties count.
  CALL {
    WITH deletedVertices
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    UNWIND deletedVertices as removedNode

    With labels(removedNode) AS nodeLabels, keys(removedNode) AS nodePropertyNames, schema
    UNWIND nodeLabels AS label
    UNWIND nodePropertyNames AS propertyName

    WITH label, propertyName, count(*) as propCount, schema
    WITH label, collect(propertyName) as propNames, collect(propCount) as propCounts, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: label})

    UNWIND range(0, size(propNames) - 1) AS index
    MATCH (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop {property_name: propNames[index]})
    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) - propCounts[index]
  }

  // Delete label property if they are less than 1 from the reductions above (zero)
  CALL {
    WITH deletedVertices
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    UNWIND deletedVertices as removedNode
    WITH labels(removedNode) AS nodeLabels, schema
    UNWIND nodeLabels AS label
    WITH DISTINCT label, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->({label: label})-[:IS_SCHEMA_NODE_PROPERTY]->(property)
    WHERE property.count < 1
    DETACH DELETE property
  }

  // reduce node out relationships count.
  CALL {
    WITH deletedVertices
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    UNWIND deletedVertices as removedNode
    WITH labels(removedNode) AS nodeLabels, schema
    UNWIND nodeLabels as nodeLabel
    WITH nodeLabel, count(*) as num_label, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->({label: nodeLabel})-[:IS_SCHEMA_NODE_OUT_REL]->(schema_node_out_rel)
    SET schema_node_out_rel.count = coalesce(schema_node_out_rel.count, 0) - num_label
  }

  // Delete node out relationship if they are less than 1 from the reductions above (zero)
  CALL {
    WITH deletedVertices
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    UNWIND deletedVertices as removedNode
    WITH labels(removedNode) AS nodeLabels, schema
    UNWIND nodeLabels as nodeLabel
    WITH DISTINCT nodeLabel, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->({label: nodeLabel})-[:IS_SCHEMA_NODE_OUT_REL]->(schema_node_out_rel)
    WHERE schema_node_out_rel.count < 1
    DETACH DELETE schema_node_out_rel
  }

  // reduce label count from deleted node.
  CALL {
    WITH deletedVertices
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    WITH deletedVertices, schema

    UNWIND deletedVertices AS removedNodes
    UNWIND labels(removedNodes) AS removedLabel
    WITH removedLabel, count(*) as num_occurances, schema

    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node {label: removedLabel})
    SET schema_node.count = schema_node.count - num_occurances
  }

  // delete nodes is there count is less than 1 from reduction above
  CALL {
    WITH deletedVertices
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH deletedVertices, schema
    UNWIND deletedVertices AS removedNodes
    UNWIND labels(removedNodes) AS removedLabel

    WITH DISTINCT removedLabel, schema
    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node {label: removedLabel})
    WHERE schema_node.count < 1
    OPTIONAL MATCH (schema_node)-->(prop_or_rel) 
    DETACH DELETE prop_or_rel, schema_node
  }

  // reduce relationship properties count
  CALL {
    WITH deletedEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    UNWIND deletedEdges as removedEdge
    WITH type(removedEdge) AS edgeType, keys(removedEdge) AS edgePropertyNames, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel {type: edgeType})

    UNWIND edgePropertyNames AS propName
    WITH edgeType, propName, count(*) as propCount, schema_rel

    MATCH (schema_rel)-->(schema_rel_prop {property_name: propName})
    SET schema_rel_prop.count = coalesce(schema_rel_prop.count, 0) - propCount
  }


  // Delete relationship property if they are less than 1 from the reductions above (zero)
  CALL {
    WITH deletedEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})
    
    UNWIND deletedEdges as removedEdge
    WITH type(removedEdge) AS edgeType, schema
    WITH DISTINCT edgeType, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->({type: edgeType})-->(property)
    WHERE property.count < 1
    DETACH DELETE property
  }


  // reduce relationship count.
  CALL {
    WITH deletedEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH deletedEdges, schema
    UNWIND deletedEdges AS deletedEdge

    WITH type(deletedEdge) AS edgeType, schema
    WITH edgeType, count(*) as num_occurances, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel {type: edgeType})
    SET schema_rel.count = coalesce(schema_rel.count, 0) - num_occurances
  }

  // delete relationships if reducutions cause it to be less than 1.
  CALL {
    WITH deletedEdges
    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})

    WITH deletedEdges, schema
    UNWIND deletedEdges AS deletedEdge

    WITH type(deletedEdge) AS edgeType, schema
    WITH DISTINCT edgeType, schema

    MATCH (schema)-[:IS_SCHEMA_RELATIONSHIP]->(schema_rel {type: edgeType})
    WHERE schema_rel.count < 1
    OPTIONAL MATCH (schema_rel)-->(rel_prop) 
    DETACH DELETE rel_prop, schema_rel
  }

}
"""

"\n// ON DELETE TRIGGER\n{\n\n  CREATE TRIGGER schema_ondelete_trigger ON DELETE\n  AFTER COMMIT EXECUTE\n  WITH deletedVertices, deletedEdges\n\n  // reduce label properties count.\n  CALL {\n    WITH deletedVertices\n    MATCH (schema :Schema{user_id: 'i9J5c8cU7NNA2cCCFYreC7'})\n    UNWIND deletedVertices as removedNode\n\n    With labels(removedNode) AS nodeLabels, keys(removedNode) AS nodePropertyNames, schema\n    UNWIND nodeLabels AS label\n    UNWIND nodePropertyNames AS propertyName\n\n    WITH label, propertyName, count(*) as propCount, schema\n    WITH label, collect(propertyName) as propNames, collect(propCount) as propCounts, schema\n\n    MATCH (schema)-[:IS_SCHEMA_NODE]->(schema_node :SchemaNode{label: label})\n\n    UNWIND range(0, size(propNames) - 1) AS index\n    MATCH (schema_node)-[:IS_SCHEMA_NODE_PROPERTY]->(schema_node_prop {property_name: propNames[index]})\n    SET schema_node_prop.count = coalesce(schema_node_prop.count, 0) - propCounts[index]\n  }\n\n  // Dele